# Analizador Morfosintáctico con HMM

Este notebook utiliza un modelo oculto de Markov (HMM) para analizar frases y etiquetarlas morfosintácticamente utilizando un corpus POS.

In [ ]:
from google.colab import files
uploaded = files.upload()
corpus_file_path = list(uploaded.keys())[0]
print("Archivo cargado:", corpus_file_path)


In [ ]:
def cargar_corpus(file_path):
    corpus = []
    with open(file_path, "r", encoding="utf-8") as file:
        for linea in file:
            linea = linea.strip()
            if linea.startswith("<doc") or linea == "":
                continue
            datos = linea.split("\t")
            if len(datos) >= 3:
                corpus.append((datos[0], datos[2]))
    return corpus

corpus = cargar_corpus(corpus_file_path)
print("Corpus cargado. Número de tokens:", len(corpus))
print("Primeros 5 registros:", corpus[:5])


In [ ]:
def calcular_probabilidades_emision(corpus):
    emision = {}
    total_etiquetas = {}
    for token, etiqueta in corpus:
        if etiqueta not in emision:
            emision[etiqueta] = {}
            total_etiquetas[etiqueta] = 0
        emision[etiqueta][token] = emision[etiqueta].get(token, 0) + 1
        total_etiquetas[etiqueta] += 1
    for etiqueta in emision:
        for token in emision[etiqueta]:
            emision[etiqueta][token] /= total_etiquetas[etiqueta]
    return emision

def calcular_probabilidades_transicion(corpus):
    transicion = {}
    total_transiciones = {}
    anterior = None
    for _, etiqueta in corpus:
        if anterior is not None:
            if anterior not in transicion:
                transicion[anterior] = {}
                total_transiciones[anterior] = 0
            transicion[anterior][etiqueta] = transicion[anterior].get(etiqueta, 0) + 1
            total_transiciones[anterior] += 1
        anterior = etiqueta
    for etiqueta in transicion:
        for siguiente in transicion[etiqueta]:
            transicion[etiqueta][siguiente] /= total_transiciones[etiqueta]
    return transicion

def calcular_probabilidades_iniciales(corpus):
    iniciales = {}
    total = 0
    if corpus:
        iniciales[corpus[0][1]] = 1
        total = 1
    for etiqueta in iniciales:
        iniciales[etiqueta] /= total
    return iniciales

emision = calcular_probabilidades_emision(corpus)
transicion = calcular_probabilidades_transicion(corpus)
estado_inicial = calcular_probabilidades_iniciales(corpus)


In [ ]:
def viterbi(frase, estados, transicion, emision, estado_inicial):
    V = [{}]
    path = {}

    for estado in estados:
        V[0][estado] = estado_inicial.get(estado, 1e-6) * emision.get(estado, {}).get(frase[0], 1e-6)
        path[estado] = [estado]

    for t in range(1, len(frase)):
        V.append({})
        new_path = {}
        for y in estados:
            (prob, estado_max) = max(
                [(V[t-1][y0] * transicion.get(y0, {}).get(y, 1e-6) * emision.get(y, {}).get(frase[t], 1e-6), y0)
                 for y0 in estados], key=lambda x: x[0])
            V[t][y] = prob
            new_path[y] = path[estado_max] + [y]
        path = new_path

    n = len(frase) - 1
    (prob, estado_max) = max([(V[n][y], y) for y in estados], key=lambda x: x[0])
    return (prob, path[estado_max])


In [ ]:
frase_prueba = ['Habla', 'con', 'el', 'enfermo', 'grave', 'de', 'trasplantes', '.']
estados = list(set([etiqueta for _, etiqueta in corpus]))

probabilidad, ruta = viterbi(frase_prueba, estados, transicion, emision, estado_inicial)
print("Frase:", frase_prueba)
print("Ruta más probable:", ruta)
print("Probabilidad total:", probabilidad)


In [ ]:
!pip install -q openpyxl
import pandas as pd

df_emision = pd.DataFrame.from_dict(emision, orient='index').fillna(0)
df_transicion = pd.DataFrame.from_dict(transicion, orient='index').fillna(0)
df_viterbi = pd.DataFrame({
    "Palabra": frase_prueba,
    "Etiqueta": ruta
})

df_emision.to_excel("tabla_emision.xlsx", engine='openpyxl')
df_transicion.to_excel("tabla_transicion.xlsx", engine='openpyxl')
df_viterbi.to_excel("ruta_viterbi.xlsx", engine='openpyxl')

from IPython.display import FileLink, display
display(FileLink("tabla_emision.xlsx"))
display(FileLink("tabla_transicion.xlsx"))
display(FileLink("ruta_viterbi.xlsx"))
